In [ ]:
from optimization import PriceOptimizer
import numpy as np
import pandas as pd

In [ ]:
def create_sample_data_with_nr_per_hl():
    """Create sample data with NR/HL information"""
    
    # Sample market data with NR/HL
    market_data = pd.DataFrame({
        'sku': [f'ABI_SKU_{i}' for i in range(10)] + [f'COMP_SKU_{i}' for i in range(5)],
        'company': ['ABI'] * 10 + ['Competitor'] * 5,
        'volume': np.random.uniform(100, 1000, 15),
        'price': np.random.uniform(20, 80, 15),
        'segment': np.random.choice(['Value', 'Core', 'Core+', 'Premium', 'Super Premium'], 15),
        'size_hierarchy': np.random.choice(['Small', 'Regular', 'Large'], 15),
        'nr_per_hl': np.random.uniform(15, 75, 15)  # NR/HL data
    })
    
    # Ensure segment hierarchy is respected in sample data
    segment_multipliers = {'Value': 0.6, 'Core': 0.8, 'Core+': 1.0, 'Premium': 1.3, 'Super Premium': 1.6}
    for idx, row in market_data.iterrows():
        if row['company'] == 'ABI':
            base_nr = 30
            market_data.loc[idx, 'nr_per_hl'] = base_nr * segment_multipliers.get(row['segment'], 1.0)
    
    # Sample elasticity data  
    all_skus = market_data['sku'].tolist()
    elasticity_data = []
    
    for target in all_skus:
        for other in all_skus:
            if target == other:
                elasticity = -1.2  # Own price elasticity
            else:
                elasticity = np.random.normal(0, 0.1)  # Cross price elasticity
            
            elasticity_data.append({
                'target_sku': target,
                'other_sku': other, 
                'elasticity': elasticity
            })
    
    elasticity_df = pd.DataFrame(elasticity_data)
    
    return market_data, elasticity_df

In [ ]:
market_data, elasticity_data = create_sample_data_with_nr_per_hl()

In [ ]:
market_data

In [ ]:
# Initialize optimizer
pinc_value = 0.03
optimizer = PriceOptimizer(
    market_data, 
    elasticity_data, 
    pinc_value, 
    pricing_multiple=5.0
)

# Run optimization WITH rounding
results = optimizer.optimize(
    learning_rate=0.01, 
    max_iterations=1000, 
    verbose=True,
    apply_rounding=True
)

In [ ]:
summary_df = optimizer.get_optimization_summary(results)

In [ ]:
summary_df